In [4]:
import pandas as pd

# Load your current cleaned dataset
df_cleaned = pd.read_csv('/content/drive/MyDrive/home-credit-default-risk/cleaned_dataset.csv')
print(f"Original shape: {df_cleaned.shape}")

features_to_drop = []

# 1. Remove meaningless SUM aggregations in time features
time_sum_features = [col for col in df_cleaned.columns if 'DAYS_' in col and '_SUM' in col]
features_to_drop.extend(time_sum_features)
print(f"Removing {len(time_sum_features)} meaningless time SUM features")

# 2. Remove meaningless SUM aggregations in ID features
id_sum_features = [col for col in df_cleaned.columns if 'SK_ID_' in col and '_SUM' in col]
features_to_drop.extend(id_sum_features)
print(f"Removing {len(id_sum_features)} meaningless ID SUM features")

# 3. Remove MIN aggregations for amount features 
amount_min_features = [col for col in df_cleaned.columns if 'AMT_' in col and '_MIN' in col]
features_to_drop.extend(amount_min_features)
print(f"Removing {len(amount_min_features)} less meaningful amount MIN features")

# 4. Remove redundant hour/timing SUM features
hour_sum_features = [col for col in df_cleaned.columns if 'HOUR_' in col and '_SUM' in col]
features_to_drop.extend(hour_sum_features)
features_to_drop.extend([col for col in df_cleaned.columns if 'SELLERPLACE_AREA' in col and '_SUM' in col])

# 5. Remove highly redundant categorical features
rare_contract_features = [col for col in df_cleaned.columns if 'NAME_CONTRACT_TYPE_XNA' in col]
features_to_drop.extend(rare_contract_features)

# 6. Remove redundant document flags
important_docs = ['FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_8']
doc_features_to_drop = [col for col in df_cleaned.columns if 'FLAG_DOCUMENT_' in col and col not in important_docs]
features_to_drop.extend(doc_features_to_drop)

# 7. Remove redundant organization types 
major_orgs = ['ORGANIZATION_TYPE_Business Entity Type 1', 'ORGANIZATION_TYPE_Government',
              'ORGANIZATION_TYPE_School', 'ORGANIZATION_TYPE_Medicine', 'ORGANIZATION_TYPE_Trade: type 1']
org_features_to_drop = [col for col in df_cleaned.columns if 'ORGANIZATION_TYPE_' in col and col not in major_orgs]
features_to_drop.extend(org_features_to_drop)

# 8. Remove redundant building information features
building_features = ['APARTMENTS', 'BASEMENTAREA', 'YEARS_BEGINEXPLUATATION', 'YEARS_BUILD',
                    'ELEVATORS', 'ENTRANCES', 'FLOORSMAX', 'FLOORSMIN', 'LANDAREA',
                    'LIVINGAPARTMENTS', 'LIVINGAREA', 'NONLIVINGAPARTMENTS']

for building in building_features:
    # average
    mode_features = [col for col in df_cleaned.columns if building in col and '_MODE' in col]
    medi_features = [col for col in df_cleaned.columns if building in col and '_MEDI' in col]
    features_to_drop.extend(mode_features)
    features_to_drop.extend(medi_features)

# 9. Remove redundant previous application features
# For previous applications, Mean and Max
prevapp_redundant = [col for col in df_cleaned.columns if 'PREVAPP_' in col and ('_MIN' in col or '_SUM' in col)]
# keep SUM for amount features
prevapp_amount_sums = [col for col in prevapp_redundant if 'AMT_' in col and '_SUM' in col]
prevapp_to_drop = [col for col in prevapp_redundant if col not in prevapp_amount_sums]
features_to_drop.extend(prevapp_to_drop)

# 10. Remove redundant POS MIN features (Monthly balance of client's previous loan
pos_redundant = [col for col in df_cleaned.columns if 'POS_' in col and '_MIN' in col]
features_to_drop.extend(pos_redundant)

# Remove duplicates
features_to_drop = list(set(features_to_drop))

print(f"\nTotal features to drop: {len(features_to_drop)}")
print(f"Features remaining: {df_cleaned.shape[1] - len(features_to_drop)}")

# Apply the feature removal
df_strategic_quick = df_cleaned.drop(columns=features_to_drop)
print(f"\nFinal shape after quick fix: {df_strategic_quick.shape}")

# Save the improved dataset
df_strategic_quick.to_csv('/content/drive/MyDrive/home-credit-default-risk/df_strategic_quick_fix.csv', index=False)
print("Strategic quick fix dataset saved!")


Original shape: (307511, 550)
Removing 10 meaningless time SUM features
Removing 3 meaningless ID SUM features
Removing 10 less meaningful amount MIN features

Total features to drop: 144
Features remaining: 406

Final shape after quick fix: (307511, 406)
Strategic quick fix dataset saved!


In [5]:
# Further refinement based on business importance
def additional_business_cleanup(df):
    additional_drops = []

    # Keep only major categories of loans
    major_purposes = ['NAME_CASH_LOAN_PURPOSE_Repairs', 'NAME_CASH_LOAN_PURPOSE_Other',
                     'NAME_CASH_LOAN_PURPOSE_Buying a new car', 'NAME_CASH_LOAN_PURPOSE_Education']
    purpose_features = [col for col in df.columns if 'NAME_CASH_LOAN_PURPOSE_' in col and col not in major_purposes]
    additional_drops.extend(purpose_features)

    # Keeps only broad categories of seller industry
    major_industries = ['NAME_SELLER_INDUSTRY_Connectivity', 'NAME_SELLER_INDUSTRY_Construction',
                       'NAME_SELLER_INDUSTRY_Consumer electronics', 'NAME_SELLER_INDUSTRY_XNA']
    industry_features = [col for col in df.columns if 'NAME_SELLER_INDUSTRY_' in col and col not in major_industries]
    additional_drops.extend(industry_features)

    # Keep major product combinations
    major_products = ['PRODUCT_COMBINATION_Cash', 'PRODUCT_COMBINATION_POS mobile with interest',
                     'PRODUCT_COMBINATION_Cash X-Sell: low']
    product_features = [col for col in df.columns if 'PRODUCT_COMBINATION_' in col and col not in major_products]
    additional_drops.extend(product_features)

    return list(set(additional_drops))

additional_drops = additional_business_cleanup(df_strategic_quick)
df_final_strategic = df_strategic_quick.drop(columns=additional_drops)

print(f"After additional cleanup: {df_final_strategic.shape}")
print(f"Total features removed: {df_cleaned.shape[1] - df_final_strategic.shape[1]}")
print(f"Reduction: {((df_cleaned.shape[1] - df_final_strategic.shape[1]) / df_cleaned.shape[1]) * 100:.1f}%")

# Save final strategic dataset
df_final_strategic.to_csv('/content/drive/MyDrive/home-credit-default-risk/df_final_strategic.csv', index=False)
print("Final strategic dataset saved!")


After additional cleanup: (307511, 364)
Total features removed: 186
Reduction: 33.8%
Final strategic dataset saved!
